# Gain financier — économies de provisionnement COBAC (H5)

**H5 (protocole, §5).** « Le système final génère des économies de provisionnement COBAC
quantifiables (≥ 20 M FCFA/an sur 500 dossiers pilotes). » Référence : règlement COBAC R-2001/07
(classification, comptabilisation et provisionnement des créances), qui fixe notamment le seuil de
**90 jours d'impayés comme critère de classement en créances en souffrance** — c'est directement
`defaut_90j`, la cible utilisée dans tous les notebooks de ce projet.

**Portée : M0 → M2, et M0 → M4 — pas M6.** Le protocole parle du « système final », donc M6.
Mais M6 n'est validé qu'à l'échelle d'un pilote à 60 dossiers (`m5_m6_llm.ipynb`), avec des résultats
non concluants (ICC test-retest sous le seuil de 0,75 pour S1/S2/S3, DeLong M4→M6 non significatif) —
calculer un gain financier dessus donnerait un chiffre à deux niveaux d'incertitude empilés (échelle
réduite + gain non validé statistiquement). **M2** est le seul palier dont le gain d'AUC sur M0/M1 est
statistiquement validé à l'échelle complète (2000 clients, `m2_m3_texte.ipynb`) ; **M4** est ajouté par
cohérence avec `shap_sousgroupe.ipynb` (échelle complète également), sachant que son gain sur M2 n'est
généralement pas significatif au test de DeLong. Le gain financier de M6 reste à calculer dans une
session ultérieure, une fois l'agent LLM relancé à une échelle statistiquement exploitable.

**Deux hypothèses non disponibles dans les données, explicitement paramétrées ci-dessous** (décision
validée avec l'utilisateur le 2026-08-23 : procéder avec des hypothèses documentées plutôt que
d'attendre les vrais chiffres) :

1. **Montant moyen de crédit par dossier.** Aucune colonne de montant emprunté dans
   `clients_synth.csv`/`transactions_synth.csv` (ce sont des données de *comportement*, pas de
   *portefeuille de crédit*). Hypothèse retenue : **1× le revenu mensuel déclaré moyen** du jeu de
   données (≈ 150 000 FCFA), un ordre de grandeur courant pour du nano-crédit — avec une analyse de
   sensibilité de 0,5× à 3× pour montrer l'effet de cette hypothèse sur le résultat.
2. **Taux de provisionnement COBAC applicable.** Le texte du règlement R-2001/07 n'a pas été consulté
   directement ici (seule la référence bibliographique et la mention du seuil de 90 jours figurent
   dans le protocole) ; **50 %** est utilisé comme valeur centrale (taux couramment cité pour les
   créances tout juste classées « douteuses », palier immédiatement au-delà du seuil de 90 jours), avec
   une sensibilité de 25 % à 100 %.

**À vérifier avant tout usage réel par la banque : ces deux valeurs sont des hypothèses de travail, pas
des chiffres extraits du règlement ou du portefeuille réel d'Afriland.** Le calcul est entièrement
paramétrique (une seule cellule de constantes ci-dessous) pour pouvoir être rejoué immédiatement avec
les vrais chiffres dès qu'ils seront disponibles.

**Méthode.** Comme les données ne contiennent que des dossiers déjà octroyés (pas de dossiers refusés
observés), le gain est simulé par un **seuil d'octroi hypothétique sur le score** — même logique que
`audit_equite` déjà utilisée dans les autres notebooks : à un taux d'approbation donné, on approuve les
dossiers les moins risqués selon le score. On compare, à volume d'octroi égal, le nombre de défauts
qui se retrouvent dans le portefeuille approuvé selon **M0** (méthode actuelle) contre selon **M2**/**M4** —
la différence, mise à l'échelle du volume annuel visé par H5 (500 dossiers), valorisée par les deux
hypothèses ci-dessus, donne le gain de provisionnement estimé.

In [ ]:
import re

import numpy as np
import pandas as pd
import torch
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.model_selection import train_test_split
from transformers import CamembertModel, CamembertTokenizer, logging as hf_logging

from categorisation import construire_parts_categories
from scoring_utils import (
    bootstrap_gain_provisionnement,
    construire_pipeline,
    construire_variables_comportementales,
    evaluer,
    grille_gain_provisionnement,
    rechercher_meilleur_C,
)

hf_logging.set_verbosity_error()  # les poids du pooler non utilisés déclenchent un avertissement sans objet ici

SEED = 42
TARGET = "defaut_90j"

DECLARATIF_NUM = ["age", "revenu_declare", "anciennete_mois"]
DECLARATIF_CAT = ["zone", "secteur", "region"]
COMPORTEMENTAL_NUM = [
    "nb_tx", "pct_debits", "inflow", "outflow", "net_flow",
    "mean_abs", "std_abs", "max_abs", "cv_abs",
    "nb_jours_actifs", "tx_par_jour", "ecart_revenu",
]
GRILLE_C = [0.001, 0.01, 0.1, 1, 10, 100]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device (embeddings CamemBERT) : {DEVICE}")

# ---- hypothèses H5 (à remplacer par les vrais chiffres dès qu'ils seront disponibles) ----
VOLUME_ANNUEL_PILOTE = 500          # dossiers/an, tel que fixé par H5 dans le protocole
MONTANT_MOYEN_CREDIT = 150_000      # FCFA — 1x le revenu mensuel déclaré moyen (voir introduction)
TAUX_PROVISIONNEMENT = 0.50         # COBAC R-2001/07, valeur centrale assumée (créances douteuses)
SEUIL_GAIN_CIBLE_H5 = 20_000_000    # FCFA/an, seuil de validation de H5
GRILLE_TAUX_APPROBATION = np.round(np.arange(0.50, 1.00, 0.05), 2)
TAUX_APPROBATION_REFERENCE = 0.80   # pour le détail ponctuel et le bootstrap

device (embeddings CamemBERT) : cuda


## Étape 1 — Charger les données

Mêmes fichiers que les notebooks précédents.

In [ ]:
clients = pd.read_csv("clients_synth.csv")
tx = pd.read_csv("transactions_synth.csv")

print(f"clients : {clients.shape[0]} lignes | transactions : {tx.shape[0]} lignes")
print(f"revenu déclaré moyen : {clients.revenu_declare.mean():,.0f} FCFA/mois "
      f"(ancre de l'hypothèse MONTANT_MOYEN_CREDIT)")

clients : 2000 lignes | transactions : 91666 lignes
revenu déclaré moyen : 148,540 FCFA/mois (ancre de l'hypothèse MONTANT_MOYEN_CREDIT)


## Étape 2 — Variables comportementales (M1, rappel)

`construire_variables_comportementales` (détaillée dans `baseline.ipynb`).

In [ ]:
comp = construire_variables_comportementales(tx, clients)
comp.head()

,client_id,nb_tx,pct_debits,inflow,outflow,mean_abs,std_abs,max_abs,nb_jours_actifs,net_flow,cv_abs,tx_par_jour,ecart_revenu
0,1,28,0.821429,1970486.0,1958456.0,140319.357143,352120.284243,1481199.0,26,12030.0,2.509421,1.076923,-5.025951
1,2,58,0.810345,4128443.0,2782051.0,119146.448276,251905.005755,1009317.0,44,1346392.0,2.114247,1.318182,-6.908895
2,3,36,0.750000,2775876.0,2040831.0,133797.416667,292261.844375,1364223.0,25,735045.0,2.184361,1.440000,-3.895725
3,4,21,0.857143,179598.0,1871038.0,97649.333333,319716.352365,1489911.0,19,-1691440.0,3.274127,1.105263,-0.814121
4,5,29,0.931034,397596.0,4884317.0,182134.931034,357177.292957,1209253.0,20,-4486721.0,1.961059,1.450000,0.181901


## Étape 3 — Reconstruire M0, M1, M2 (rappel condensé)

Reprise résumée de `baseline.ipynb` (M0, M1) et `m2_m3_texte.ipynb` (M2) — même split
(`SEED=42, test_size=0.20`), mêmes clients en test que dans tous les autres notebooks.

In [ ]:
parts_m2, tx_categorise_m2 = construire_parts_categories(tx)
PART_COLS_M2 = [c for c in parts_m2.columns if c.startswith("PART_")]

df_m2 = (clients
         .merge(comp, on="client_id", how="left")
         .merge(parts_m2, on="client_id", how="left"))
df_m2[COMPORTEMENTAL_NUM + PART_COLS_M2] = df_m2[COMPORTEMENTAL_NUM + PART_COLS_M2].fillna(0)

y = df_m2[TARGET].values
colonnes_gt = [c for c in df_m2.columns if c.startswith("gt_")]
X_m2 = df_m2.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr_m2, Xte_m2, ytr_m2, yte_m2 = train_test_split(
    X_m2, y, test_size=0.20, stratify=y, random_state=SEED
)
print(f"train {len(Xtr_m2)} | test {len(Xte_m2)} | taux de défaut test : {yte_m2.mean():.3f}")

train 1600 | test 400 | taux de défaut test : 0.158


In [ ]:
pipeline_m0 = construire_pipeline(DECLARATIF_NUM, DECLARATIF_CAT, seed=SEED)
modele_m0, _ = rechercher_meilleur_C(pipeline_m0, Xtr_m2, ytr_m2, GRILLE_C, seed=SEED)
p0 = modele_m0.predict_proba(Xte_m2)[:, 1]
_ = evaluer("M0", yte_m2, p0)

num_m1 = DECLARATIF_NUM + COMPORTEMENTAL_NUM
pipeline_m1 = construire_pipeline(num_m1, DECLARATIF_CAT, seed=SEED)
modele_m1, _ = rechercher_meilleur_C(pipeline_m1, Xtr_m2, ytr_m2, GRILLE_C, seed=SEED)
p1 = modele_m1.predict_proba(Xte_m2)[:, 1]
_ = evaluer("M1", yte_m2, p1)

num_m2 = num_m1 + PART_COLS_M2
pipeline_m2 = construire_pipeline(num_m2, DECLARATIF_CAT, seed=SEED)
modele_m2, _ = rechercher_meilleur_C(pipeline_m2, Xtr_m2, ytr_m2, GRILLE_C, seed=SEED)
p2 = modele_m2.predict_proba(Xte_m2)[:, 1]
_ = evaluer("M2", yte_m2, p2)

M0   | AUC 0.653 | Gini 0.307 | KS 0.268


M1   | AUC 0.666 | Gini 0.333 | KS 0.275


M2   | AUC 0.718 | Gini 0.436 | KS 0.351


## Étape 4 — Reconstruire M4 (embeddings + clustering, rappel condensé)

Reprise résumée de `m4_embeddings.ipynb` (déjà revalidée à l'échelle complète dans
`shap_sousgroupe.ipynb`) : nettoyage léger, embeddings CamemBERT par mean-pooling, PCA puis K-Means
(K choisi par silhouette).

In [ ]:
def nettoyer_pour_embedding(libelle):
    """Nettoyage léger : garde les mots et leur ordre, retire le bruit technique."""
    s = str(libelle).lower()
    s = re.sub(r"(ref|tpe|ag)\w*", " ", s)
    s = re.sub(r"\d+", " ", s)
    s = re.sub(r"[^a-zàâäéèêëïîôöùûüç\s]", " ", s)
    return re.sub(r"\s+", " ", s).strip()


tokenizer = CamembertTokenizer.from_pretrained("camembert-base")
modele_embed = CamembertModel.from_pretrained("camembert-base").to(DEVICE).eval()


def embarquer_textes(textes, batch_size=64):
    """Embedding de phrase par mean-pooling des dernières couches cachées de CamemBERT."""
    vecteurs = []
    for i in range(0, len(textes), batch_size):
        lot = textes[i:i + batch_size]
        entrees = tokenizer(lot, padding=True, truncation=True, max_length=64,
                             return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            sortie = modele_embed(**entrees).last_hidden_state
        masque = entrees["attention_mask"].unsqueeze(-1).float()
        moyenne = (sortie * masque).sum(1) / masque.sum(1).clamp(min=1e-9)
        vecteurs.append(moyenne.cpu().numpy())
    return np.vstack(vecteurs)


tx["libelle_net"] = tx.libelle.map(nettoyer_pour_embedding)
uniques_df = pd.DataFrame({"libelle_net": tx.libelle_net.drop_duplicates().reset_index(drop=True)})
embeddings_uniques = embarquer_textes(uniques_df.libelle_net.tolist())
print(f"{len(uniques_df)} libellés uniques (nettoyés) sur {len(tx)} transactions")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

33040 libellés uniques (nettoyés) sur 91666 transactions


In [ ]:
pca = PCA(n_components=50, random_state=SEED)
embeddings_reduits = pca.fit_transform(embeddings_uniques)

rng = np.random.default_rng(SEED)
grille_K = [5, 8, 10, 12, 15, 18, 20, 25]
resultats_K = []
for k in grille_K:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10).fit(embeddings_reduits)
    if len(embeddings_reduits) > 3000:
        idx = rng.choice(len(embeddings_reduits), 3000, replace=False)
        sil = silhouette_score(embeddings_reduits[idx], km.labels_[idx])
    else:
        sil = silhouette_score(embeddings_reduits, km.labels_)
    resultats_K.append({"K": k, "inertie": km.inertia_, "silhouette": sil})

resultats_K = pd.DataFrame(resultats_K)
K_FINAL = int(resultats_K.loc[resultats_K.silhouette.idxmax(), "K"])
print(f"K retenu (meilleur silhouette) : {K_FINAL}")

kmeans_final = KMeans(n_clusters=K_FINAL, random_state=SEED, n_init=10).fit(embeddings_reduits)
uniques_df["cluster"] = kmeans_final.labels_

tx_clusterise = tx.merge(uniques_df, on="libelle_net", how="left")
parts_clust = (tx_clusterise.groupby(["client_id", "cluster"]).size()
               .unstack(fill_value=0))
parts_clust = parts_clust.div(parts_clust.sum(axis=1), axis=0)
parts_clust.columns = [f"PART_CLUST_{c}" for c in parts_clust.columns]
parts_clust = parts_clust.reset_index()
PART_CLUST_COLS = [c for c in parts_clust.columns if c.startswith("PART_CLUST_")]

K retenu (meilleur silhouette) : 5


In [ ]:
df_m4 = (clients
         .merge(comp, on="client_id", how="left")
         .merge(parts_m2, on="client_id", how="left")
         .merge(parts_clust, on="client_id", how="left"))
colonnes_a_remplir = COMPORTEMENTAL_NUM + PART_COLS_M2 + PART_CLUST_COLS
df_m4[colonnes_a_remplir] = df_m4[colonnes_a_remplir].fillna(0)

y4 = df_m4[TARGET].values
colonnes_gt = [c for c in df_m4.columns if c.startswith("gt_")]
X_m4 = df_m4.drop(columns=colonnes_gt + [TARGET, "client_id"])

Xtr_m4, Xte_m4, ytr_m4, yte_m4 = train_test_split(
    X_m4, y4, test_size=0.20, stratify=y4, random_state=SEED
)
# même split que M0/M1/M2 (même ordre de clients, même SEED/test_size) : vérifié avant de
# réutiliser p0 tel quel contre yte_m4 plus bas
assert (yte_m4 == yte_m2).all(), "le split M4 ne correspond pas au split M0/M1/M2 attendu"

num_m4 = DECLARATIF_NUM + COMPORTEMENTAL_NUM + PART_CLUST_COLS
pipeline_m4 = construire_pipeline(num_m4, DECLARATIF_CAT, seed=SEED)
modele_m4, _ = rechercher_meilleur_C(pipeline_m4, Xtr_m4, ytr_m4, GRILLE_C, seed=SEED)
p4 = modele_m4.predict_proba(Xte_m4)[:, 1]
_ = evaluer("M4", yte_m4, p4)

M4   | AUC 0.681 | Gini 0.361 | KS 0.297


## Étape 5 — Grille de gain financier, M0 → M2 et M0 → M4

Pour chaque taux d'approbation de la grille, `grille_gain_provisionnement` compare le nombre de
défauts qui se retrouveraient dans le portefeuille approuvé selon M0 contre selon M2 (puis M4) — la
différence, mise à l'échelle de `VOLUME_ANNUEL_PILOTE`, valorisée par les deux hypothèses. La ligne
`taux_approbation = {TAUX_APPROBATION_REFERENCE}` sert de référence ponctuelle pour le bootstrap de
l'étape suivante.

In [ ]:
gain_m0_m2 = grille_gain_provisionnement(
    yte_m2, p0, p2, GRILLE_TAUX_APPROBATION,
    MONTANT_MOYEN_CREDIT, TAUX_PROVISIONNEMENT, VOLUME_ANNUEL_PILOTE,
)
gain_m0_m2["atteint_seuil_H5"] = gain_m0_m2.gain_fcfa_an >= SEUIL_GAIN_CIBLE_H5
gain_m0_m2

,taux_approbation,defauts_evites_test,defauts_evites_annuels_estimes,gain_fcfa_an,atteint_seuil_H5
0,0.50,4,10.000000,7.500000e+05,False
1,0.55,4,9.090909,6.818182e+05,False
2,0.60,7,14.583333,1.093750e+06,False
3,0.65,6,11.538462,8.653846e+05,False
4,0.70,6,10.714286,8.035714e+05,False
5,0.75,6,10.000000,7.500000e+05,False
6,0.80,2,3.125000,2.343750e+05,False
7,0.85,1,1.470588,1.102941e+05,False
8,0.90,1,1.388889,1.041667e+05,False
9,0.95,3,3.947368,2.960526e+05,False


In [ ]:
gain_m0_m4 = grille_gain_provisionnement(
    yte_m2, p0, p4, GRILLE_TAUX_APPROBATION,
    MONTANT_MOYEN_CREDIT, TAUX_PROVISIONNEMENT, VOLUME_ANNUEL_PILOTE,
)
gain_m0_m4["atteint_seuil_H5"] = gain_m0_m4.gain_fcfa_an >= SEUIL_GAIN_CIBLE_H5
gain_m0_m4

,taux_approbation,defauts_evites_test,defauts_evites_annuels_estimes,gain_fcfa_an,atteint_seuil_H5
0,0.50,0,0.000000,0.000000,False
1,0.55,3,6.818182,511363.636364,False
2,0.60,3,6.250000,468750.000000,False
3,0.65,5,9.615385,721153.846154,False
4,0.70,5,8.928571,669642.857143,False
5,0.75,3,5.000000,375000.000000,False
6,0.80,2,3.125000,234375.000000,False
7,0.85,-1,-1.470588,-110294.117647,False
8,0.90,-1,-1.388889,-104166.666667,False
9,0.95,1,1.315789,98684.210526,False


## Étape 6 — Incertitude d'échantillonnage (bootstrap) au taux de référence

`bootstrap_gain_provisionnement` rééchantillonne le jeu de test 2000 fois pour donner un intervalle
de confiance à 95 % sur le gain, au taux d'approbation de référence — dans le même esprit que le test
de DeLong pour les AUC, mais ici pour le gain financier dérivé (pas un test d'hypothèse formel, un
intervalle de confiance).

In [ ]:
gain_moyen_m2, ic_m2 = bootstrap_gain_provisionnement(
    yte_m2, p0, p2, TAUX_APPROBATION_REFERENCE,
    MONTANT_MOYEN_CREDIT, TAUX_PROVISIONNEMENT, VOLUME_ANNUEL_PILOTE,
)
gain_moyen_m4, ic_m4 = bootstrap_gain_provisionnement(
    yte_m2, p0, p4, TAUX_APPROBATION_REFERENCE,
    MONTANT_MOYEN_CREDIT, TAUX_PROVISIONNEMENT, VOLUME_ANNUEL_PILOTE,
)

print(f"M0->M2 à {TAUX_APPROBATION_REFERENCE:.0%} d'approbation : "
      f"{gain_moyen_m2:,.0f} FCFA/an [IC 95% {ic_m2[0]:,.0f} ; {ic_m2[1]:,.0f}]")
print(f"M0->M4 à {TAUX_APPROBATION_REFERENCE:.0%} d'approbation : "
      f"{gain_moyen_m4:,.0f} FCFA/an [IC 95% {ic_m4[0]:,.0f} ; {ic_m4[1]:,.0f}]")
print(f"seuil H5 : {SEUIL_GAIN_CIBLE_H5:,.0f} FCFA/an")

M0->M2 à 80% d'approbation : 249,434 FCFA/an [IC 95% -585,938 ; 1,054,688]
M0->M4 à 80% d'approbation : 244,043 FCFA/an [IC 95% -351,562 ; 820,312]
seuil H5 : 20,000,000 FCFA/an


## Étape 7 — Analyse de sensibilité aux deux hypothèses

Le gain (M0 → M2, au taux d'approbation de référence) recalculé sur une grille de multiples du revenu
mensuel déclaré (montant de crédit) et de taux de provisionnement COBAC — pour rendre visible l'effet
des deux hypothèses de l'introduction sur le résultat, plutôt que de le cacher derrière un seul
chiffre.

In [ ]:
revenu_moyen = clients.revenu_declare.mean()
lignes_sensi = []
for multiple_montant in [0.5, 1.0, 2.0, 3.0]:
    for taux_prov in [0.25, 0.50, 0.75, 1.00]:
        montant = multiple_montant * revenu_moyen
        gain, ic = bootstrap_gain_provisionnement(
            yte_m2, p0, p2, TAUX_APPROBATION_REFERENCE, montant, taux_prov, VOLUME_ANNUEL_PILOTE,
            n_boot=500,  # grille de sensibilité : moins de tirages, juste pour situer l'ordre de grandeur
        )
        lignes_sensi.append({
            "montant_credit_fcfa": round(montant),
            "taux_provisionnement": taux_prov,
            "gain_fcfa_an": round(gain),
            "atteint_seuil_H5": gain >= SEUIL_GAIN_CIBLE_H5,
        })

sensi = pd.DataFrame(lignes_sensi)
sensi

,montant_credit_fcfa,taux_provisionnement,gain_fcfa_an,atteint_seuil_H5
0,74270,0.25,63710,False
1,74270,0.50,127419,False
2,74270,0.75,191129,False
3,74270,1.00,254839,False
4,148540,0.25,127419,False
5,148540,0.50,254839,False
6,148540,0.75,382258,False
7,148540,1.00,509678,False
8,297080,0.25,254839,False
9,297080,0.50,509678,False


## Lecture des résultats

**À lire une fois les cellules ci-dessus exécutées.** Le gain estimé (M0 → M2, hypothèses centrales)
se lit à l'Étape 6, comparé au seuil H5 de 20 M FCFA/an ; l'Étape 7 montre à quel point ce verdict
dépend des deux hypothèses non mesurées (montant de crédit, taux de provisionnement) plutôt que de le
présenter comme un chiffre unique et définitif.

**Limites assumées :**
- **Pas de données de refus réelles.** Le portefeuille observé est déjà entièrement octroyé (tous les
  clients de `clients_synth.csv` ont un crédit, avec ou sans défaut) — la simulation par seuil de score
  traite ce portefeuille comme un vivier de candidats hypothétique, à la manière de `audit_equite` dans
  les autres notebooks. C'est une approximation standard en évaluation de scoring (comparer un nouveau
  score à volume d'octroi égal), mais elle suppose que la distribution des candidats réels ressemblerait
  à celle du portefeuille déjà accepté — pas garanti si le système actuel présente déjà un biais de
  sélection à l'entrée.
- **`MONTANT_MOYEN_CREDIT` et `TAUX_PROVISIONNEMENT` sont des hypothèses de travail**, pas des chiffres
  extraits du portefeuille réel d'Afriland ni du texte du règlement COBAC R-2001/07 (non consulté
  directement) — voir l'introduction et l'analyse de sensibilité de l'Étape 7. À remplacer dès que les
  vrais chiffres seront disponibles ; le calcul (Étapes 5-7) est entièrement paramétrique.
- **M6 non couvert** (voir l'introduction) — le calcul suppose implicitement que M2 (ou M4) est le
  « système final » à évaluer, faute d'un M6 validé à l'échelle complète.
- **`VOLUME_ANNUEL_PILOTE = 500`** reprend tel quel le chiffre du protocole (H5), pas un volume observé
  ou négocié avec la banque.
- **Intervalle de confiance bootstrap (Étape 6)** quantifie l'incertitude d'échantillonnage sur le jeu
  de test (~400 clients), pas l'incertitude sur les deux hypothèses elles-mêmes (couverte séparément,
  de façon non probabiliste, par l'analyse de sensibilité).